<a id="top"></a>

# Lab L1004: build the agent loop (termination)

```yaml
title: "Lab L1004: build the agent loop (termination)"
keywords: agent loop, run_loop, stub model, natural termination, max_steps cap, message history invariant, multiple tool_use, offline, lab
estimated duration: 40
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md). Solutions: `L1004_lab_solutions.ipynb`. **Pure Python — no API key needed.** You build the loop against a *scripted stub model*, so it runs the same way every time (the same offline approach as the [L1003 demo](L1003_lecture.ipynb)).
>
> **After this lab you can:** write a model→tool→model loop that maintains the message-history invariant, terminates naturally when the model stops calling tools, and halts on a `max_steps` cap when it doesn't.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Detect natural termination](#2-problem-1--detect-natural-termination)
- [3. Problem 2 — Write run_loop](#3-problem-2--write-run_loop)
- [4. Problem 3 — Drive it: natural termination](#4-problem-3--drive-it-natural-termination)
- [5. Problem 4 — Drive it: the max_steps cap catches a runaway](#5-problem-4--drive-it-the-max_steps-cap-catches-a-runaway)
- [6. Problem 5 — Two tool calls in one response (written)](#6-problem-5--two-tool-calls-in-one-response-written)
- [7. Self-check](#7-self-check)

## 1. Setup

All given: the `calculator` + `lookup` tools and a dispatch table; a `FakeModel` whose `.create(...)` pops the next *scripted* response (mimicking the SDK's `.content` blocks); a `dispatch` helper that runs one tool and returns a `tool_result`; and the `RunResult` dataclass your loop will return. You write **only** the loop.

In [1]:
from types import SimpleNamespace


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# Dispatch table: tool NAME -> the function that runs it.
TOOLS: dict[str, object] = {"calculator": calculator, "lookup": lookup}

In [2]:
def text_block(s: str) -> SimpleNamespace:
    """A fake assistant text block."""
    return SimpleNamespace(type="text", text=s)


def tool_use_block(call_id: str, name: str, args: dict[str, object]) -> SimpleNamespace:
    """A fake assistant tool_use block (mimics the SDK's tool_use content block)."""
    return SimpleNamespace(type="tool_use", id=call_id, name=name, input=args)


def response(blocks: list[SimpleNamespace]) -> SimpleNamespace:
    """A fake model response."""
    stop = "tool_use" if any(b.type == "tool_use" for b in blocks) else "end_turn"
    return SimpleNamespace(content=blocks, stop_reason=stop)


class FakeModel:
    """A scripted stand-in for the Anthropic client: .create() pops the next response."""

    def __init__(self, scripted: list[SimpleNamespace]) -> None:
        self._scripted = scripted
        self.calls = 0

    def create(self, **kwargs: object) -> SimpleNamespace:
        # If the script runs out, reuse the last line -- this is how we simulate a runaway.
        i = min(self.calls, len(self._scripted) - 1)
        self.calls += 1
        return self._scripted[i]

In [3]:
def dispatch(tools: dict[str, object], call: object) -> dict[str, object]:
    """Run one requested tool and return a tool_result block (errors become is_error results)."""
    fn = tools.get(call.name)
    if fn is None:
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": f"no such tool {call.name!r}",
            "is_error": True,
        }
    try:
        output = fn(**call.input)
        return {"type": "tool_result", "tool_use_id": call.id, "content": output}
    except Exception as exc:
        return {
            "type": "tool_result",
            "tool_use_id": call.id,
            "content": repr(exc),
            "is_error": True,
        }

In [4]:
from dataclasses import dataclass


@dataclass
class RunResult:
    """The answer, the number of model calls, and WHY the loop stopped."""

    final_text: str
    iterations: int
    termination: str  # "natural" or "max_steps"

[↑ Back to top](#top)

## 2. Problem 1 — Detect natural termination

Warm-up before the full loop. Write `is_done(resp)` that returns `True` when a model response has **no `tool_use` block** (only `text`) — that is *natural termination*, the one signal that means 'the model thinks it's done.' A response is done iff none of its `resp.content` blocks have `.type == "tool_use"`.

In [5]:
def is_done(resp: object) -> bool:
    """Natural termination: the response carries no tool_use block."""
    return not any(b.type == "tool_use" for b in resp.content)


print(is_done(response([text_block("all done")])))  # expect True
print(is_done(response([tool_use_block("c1", "lookup", {"city": "Paris"})])))  # expect False

True
False


[↑ Back to top](#top)

## 3. Problem 2 — Write run_loop

Now the whole loop. `run_loop(model, tools, user_msg, max_steps)` should, for up to `max_steps` iterations:

1. Call `model.create(model=..., max_tokens=..., tools=list(tools), messages=messages)`.
2. Collect the `tool_use` blocks. If there are **none**, join the `text` blocks and return `RunResult(final_text, step, "natural")`.
3. Otherwise — **Rule 1, the message-history invariant** — append the assistant turn (`{"role": "assistant", "content": resp.content}`), then run **every** `tool_use` via `dispatch(tools, call)`, collect all the `tool_result`s, and append them as **one** user-role message before looping.
4. If the loop runs out of steps, return `RunResult("", max_steps, "max_steps")`.

Add a `print()` per iteration (iteration number + the tool calls) — your minimum-viable trace for L11.

In [6]:
def run_loop(model: object, tools: dict[str, object], user_msg: str, max_steps: int) -> RunResult:
    messages: list[dict[str, object]] = [{"role": "user", "content": user_msg}]
    for step in range(1, max_steps + 1):
        resp = model.create(
            model="claude-sonnet-4-6", max_tokens=512, tools=list(tools), messages=messages
        )
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            final_text = "".join(b.text for b in resp.content if b.type == "text")
            print(f"[step {step}] natural termination")
            return RunResult(final_text, step, "natural")
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for call in tool_uses:
            print(f"[step {step}] tool_use: {call.name}({call.input})")
            results.append(dispatch(tools, call))
        messages.append({"role": "user", "content": results})
    print(f"[stop] max_steps={max_steps} hit")
    return RunResult("", max_steps, "max_steps")

[↑ Back to top](#top)

## 4. Problem 3 — Drive it: natural termination

Script a stub that calls `calculator`, then `lookup`, then answers in plain text. Run `run_loop` with `max_steps=10` and confirm it terminates **naturally** in 3 iterations.

In [7]:
happy_script = [
    response([tool_use_block("c1", "calculator", {"expression": "36 * 1"})]),
    response([tool_use_block("c2", "lookup", {"city": "Lagos"})]),
    response([text_block("Lagos has about 15,000,000 people.")]),
]
model = FakeModel(happy_script)
result = run_loop(model, TOOLS, "Population of Lagos?", max_steps=10)
print("\nfinal_text :", result.final_text)
print("iterations :", result.iterations)
print("termination:", result.termination)
assert result.termination == "natural" and result.iterations == 3

[step 1] tool_use: calculator({'expression': '36 * 1'})
[step 2] tool_use: lookup({'city': 'Lagos'})
[step 3] natural termination

final_text : Lagos has about 15,000,000 people.
iterations : 3
termination: natural


[↑ Back to top](#top)

## 5. Problem 4 — Drive it: the max_steps cap catches a runaway

Script a stub that **never stops** — one `lookup` call that the `FakeModel` will repeat forever. Run `run_loop` with `max_steps=5` and confirm the **cap** fires (termination `"max_steps"`, 5 iterations). This is the safety net: a loop with no cap is broken, not minimal.

In [8]:
runaway_script = [response([tool_use_block("r1", "lookup", {"city": "Tokyo"})])]
model = FakeModel(runaway_script)
result = run_loop(model, TOOLS, "Keep checking Tokyo.", max_steps=5)
print("\ntermination:", result.termination, "  <- the cap fired, not the model")
assert result.termination == "max_steps" and result.iterations == 5

[step 1] tool_use: lookup({'city': 'Tokyo'})
[step 2] tool_use: lookup({'city': 'Tokyo'})
[step 3] tool_use: lookup({'city': 'Tokyo'})
[step 4] tool_use: lookup({'city': 'Tokyo'})
[step 5] tool_use: lookup({'city': 'Tokyo'})
[stop] max_steps=5 hit

termination: max_steps   <- the cap fired, not the model


[↑ Back to top](#top)

## 6. Problem 5 — Two tool calls in one response (written)

A single model response can contain **more than one** `tool_use` block. In a sentence or two: what must your loop do with all of them *before* it calls the model again, and why? (Hint: the message-history invariant — Rule 1.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1004_lab_solutions.ipynb`. Done when: `is_done` returns `True`/`False` correctly; the happy run terminates `"natural"` in 3 iterations; the runaway run terminates `"max_steps"` in 5; and you can say what the loop does with multiple `tool_use` blocks (run all of them, return all their `tool_result`s in one user message).

[↑ Back to top](#top)